[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Optimizer077/jev-learning-course/blob/main/notebooks/05_optional_real_jev_api.ipynb)

# 5 · Optional: call the real Jev API

**Path:** Optional integration  
**Before you start:** Lesson 01; Python dictionaries; an API key only for the live call

[Course home](../README.md) · [Course outline](../docs/COURSE.md) · [Setup help](../docs/SETUP.md) · [Glossary](../docs/GLOSSARY.md)

## The idea before the code

An API is a way for your program to ask another service for a result. First inspect the request and response locally; enable the real request only when you want it.

**Your first pass:** Keep RUN_LIVE = False on your first pass. The long validator is reference code: understand its purpose before every line.

### Words you need for this lesson

| Word | Everyday meaning |
|---|---|
| **API** | A way for programs to ask a service for a result. |
| **Request** | The information sent to that service. |
| **Fixture** | A handmade example used to check code locally. |

## Goal
Inspect one real request and validate the response contract. **The saved run is offline.**
It validates handmade fixtures; it contains no live Jev result.

## Setup
You need an API key from the [official console](https://console.typesafe.ai/) only for the live step.
This uses Python's standard HTTP library and the [official API](https://docs.typesafe.ai/api).
Live usage may be billed. The only data sent are the fictional state and questions shown below.

The model is pinned to `jev-1.13.0`, listed when documentation was checked on 2026-10-01.
Aliases can move; check [Models](https://docs.typesafe.ai/models) if that version becomes unavailable.
The model card currently describes text input, including structured text, rather than image/audio input.

In [1]:
# Find the included teaching helpers from the course folder.
from pathlib import Path
import sys
REQUIRED_FILES = ("src/lab_core.py", "src/tutorial_utils.py", "data/tickets.json",
                  "assets/decision-flow.png", "assets/question-types.png",
                  "assets/calibration-counts.png", "assets/data-splits.png", "assets/word-order.png")
COURSE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if all((p / name).is_file() for name in REQUIRED_FILES)), None)
# Colab opens a single notebook; fetch its companion code and fictional data.
if COURSE_ROOT is None:
    try:
        import google.colab
    except ImportError:
        pass
    else:
        import subprocess
        COURSE_ROOT = Path("/content/jev-learning-course")
        if COURSE_ROOT.exists() and not all((COURSE_ROOT / name).is_file() for name in REQUIRED_FILES):
            raise FileNotFoundError("The cached Colab course is incomplete. Start a fresh runtime and run all again.")
        if not COURSE_ROOT.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Optimizer077/jev-learning-course.git",
                            str(COURSE_ROOT)], check=True)
if COURSE_ROOT is None:
    raise FileNotFoundError("Extract the complete course and open it from its folder.")
if str(COURSE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT / "src"))

import json
import math
import os
from getpass import getpass
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

RUN_LIVE = False  # Set True yourself to send one request.
MODEL = 'jev-1.13.0'
ENDPOINT = 'https://api.typesafe.ai/v1/systemone'

## Steps
### 1. Build the request
Each answer is returned under its question ID. Meaning belongs in `instructions` and `criteria`;
the question ID is a lookup key, not a substitute for asking the question.

In [2]:
payload = {
    'model': MODEL,
    'state': {'customer_message': 'My CSV export crashes in every browser. I cannot finish my report.'},
    'questions': {
        'queue': {
            'type': 'choice',
            'instructions': 'Which support queue best matches the customer_message?',
            'criteria': {
                'technical': 'Broken application behavior',
                'billing': 'Payments, invoices, or refunds',
                'other': 'Neither listed topic applies, or information is insufficient',
            },
        },
        'disruption': {
            'type': 'score',
            'instructions': 'How much does the reported issue disrupt the stated task?',
            'criteria': ['No disruption', 'Some disruption but task can continue', 'Task is blocked'],
        },
        'refund_requested': {
            'type': 'noul',
            'instructions': 'Does customer_message explicitly ask for a refund?',
        },
    },
}
print(json.dumps(payload, indent=2))

{
  "model": "jev-1.13.0",
  "state": {
    "customer_message": "My CSV export crashes in every browser. I cannot finish my report."
  },
  "questions": {
    "queue": {
      "type": "choice",
      "instructions": "Which support queue best matches the customer_message?",
      "criteria": {
        "technical": "Broken application behavior",
        "billing": "Payments, invoices, or refunds",
        "other": "Neither listed topic applies, or information is insufficient"
      }
    },
    "disruption": {
      "type": "score",
      "instructions": "How much does the reported issue disrupt the stated task?",
      "criteria": [
        "No disruption",
        "Some disruption but task can continue",
        "Task is blocked"
      ]
    },
    "refund_requested": {
      "type": "noul",
      "instructions": "Does customer_message explicitly ask for a refund?"
    }
  }
}


### 2. Check the response where it enters your application
The following checks cover the fields this tutorial uses. They check structure and arithmetic,
not whether a judgment is correct. The tolerance allows minor floating-point differences.

In [3]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)

def probability(value):
    return finite_number(value) and 0 <= value <= 1

def validate_response(response, request):
    require(isinstance(response, dict), 'Response must be an object')
    require(isinstance(response.get('model'), str), 'Missing model identity')
    answers = response.get('answers')
    require(isinstance(answers, dict), 'Missing answers object')
    require(set(answers) == set(request['questions']), 'Answer IDs do not match questions')
    for name, question in request['questions'].items():
        answer = answers[name]
        require(isinstance(answer, dict), f'{name}: answer must be an object')
        kind = question['type']
        require(answer.get('type') == kind, f'{name}: unexpected answer type')
        if kind == 'noul':
            require(probability(answer.get('noul')), f'{name}: invalid Noul')
            continue
        distribution = answer.get('probabilities')
        require(isinstance(distribution, dict), f'{name}: missing distribution')
        expected = (set(question['criteria']) if kind == 'choice'
                    else {str(k) for k in range(len(question['criteria']))})
        require(set(distribution) == expected, f'{name}: unexpected options')
        require(all(probability(v) for v in distribution.values()), f'{name}: invalid probability')
        require(math.isclose(sum(distribution.values()), 1, abs_tol=1e-5), f'{name}: invalid sum')
        require(probability(answer.get('confidence')), f'{name}: invalid confidence')
        if kind == 'choice':
            selected = answer.get('choice')
            require(selected in expected, f'{name}: invalid choice')
            require(math.isclose(distribution[selected], max(distribution.values()), abs_tol=1e-5),
                    f'{name}: choice is not a highest-probability option')
        else:
            score = answer.get('score')
            require(finite_number(score), f'{name}: invalid score')
            expected_score = sum(int(k) * v for k, v in distribution.items())
            require(math.isclose(score, expected_score, abs_tol=1e-5), f'{name}: inconsistent score')
            legend = answer.get('legend')
            require(isinstance(legend, dict) and set(legend) == expected, f'{name}: invalid legend')
    return answers

### 3. Exercise the checks without calling a model
This fixture is entirely handmade, including the confidence values. It is not a cached API response,
and does not demonstrate TypeSafe's confidence calculation.

In [4]:
fixture = {
    'model': 'HANDMADE-TEACHING-FIXTURE',
    'answers': {
        'queue': {'type': 'choice', 'choice': 'technical',
                  'probabilities': {'technical': 0.90, 'billing': 0.03, 'other': 0.07},
                  'confidence': 0.80},
        'disruption': {'type': 'score', 'score': 1.75,
                       'probabilities': {'0': 0.05, '1': 0.15, '2': 0.80},
                       'legend': {str(i): label for i, label in enumerate(payload['questions']['disruption']['criteria'])},
                       'confidence': 0.65},
        'refund_requested': {'type': 'noul', 'noul': 0.04},
    },
}
validated = validate_response(fixture, payload)
print('Handmade fixture passed local checks. No API request was made.')

Handmade fixture passed local checks. No API request was made.


### 4. Enable one live request when ready
The key is read from the environment or requested with hidden input. It is never printed.
No network call occurs with `RUN_LIVE = False`. This small teaching client has a timeout and stops
on HTTP failures; for repeated workloads use the official SDK's retry handling.

In [5]:
def call_jev(request_payload, api_key):
    request = Request(
        ENDPOINT, data=json.dumps(request_payload).encode('utf-8'), method='POST',
        headers={'Authorization': 'Bearer ' + api_key, 'Content-Type': 'application/json'},
    )
    try:
        with urlopen(request, timeout=45) as result:
            return json.load(result)
    except HTTPError as error:
        hints = {401: 'Check the API key.', 422: 'Check request fields and current docs.',
                 429: 'Rate limited; wait before retrying.', 529: 'Service overloaded; retry later.'}
        raise RuntimeError(f'HTTP {error.code}. ' + hints.get(error.code, 'Check the service status.')) from None
    except (URLError, TimeoutError):
        raise RuntimeError('Network request failed or timed out; check connectivity.') from None

live_response = None
if RUN_LIVE:
    key = os.environ.get('TYPESAFE_API_KEY') or getpass('TypeSafe API key (hidden): ')
    try:
        if not key.strip():
            raise ValueError('An API key is required for a live request.')
        live_response = call_jev(payload, key.strip())
        live_answers = validate_response(live_response, payload)
        print('Actual responding model:', live_response['model'])
        print(json.dumps(live_answers, indent=2))
    finally:
        del key
else:
    print('LIVE REQUEST SKIPPED. Set RUN_LIVE = True and rerun to call Jev.')

LIVE REQUEST SKIPPED. Set RUN_LIVE = True and rerun to call Jev.


> **Optional deeper practice.** You can stop reading here on your first pass. If you run the notebook, use Run All so the later checks still have their inputs.

### 5. Inspect the complete decision without losing uncertainty
This cell works with the explicit teaching fixture in offline mode. It shows the **selected probability**
separately from the **API confidence** so they cannot be accidentally treated as the same field.
The confidence value shown offline remains invented.

In [6]:
from tutorial_utils import table
active_response = live_response if live_response is not None else fixture
active_answers = validate_response(active_response, payload)
queue = active_answers['queue']
table(['Source', 'Choice', 'Selected probability', 'Confidence'], [[
    'LIVE' if live_response is not None else 'HANDMADE FIXTURE',
    queue['choice'], queue['probabilities'][queue['choice']], queue['confidence']]])

Source,Choice,Selected probability,Confidence
HANDMADE FIXTURE,technical,0.9,0.8


### 6. Test several failure modes locally
An integration should fail visibly on a broken contract. Transport success alone is insufficient.
These mutations exercise different invariants, without contacting the API.

In [7]:
import copy
mutations = {
    'missing question': lambda r: r['answers'].pop('queue'),
    'unknown option': lambda r: r['answers']['queue'].update(choice='invented_queue'),
    'bad sum': lambda r: r['answers']['queue']['probabilities'].update(technical=.5),
    'nonfinite value': lambda r: r['answers']['refund_requested'].update(noul=float('nan')),
    'incorrect expectation': lambda r: r['answers']['disruption'].update(score=.1),
}
rejected = []
for label, mutate in mutations.items():
    example = copy.deepcopy(fixture)
    mutate(example)
    try:
        validate_response(example, payload)
    except ValueError:
        rejected.append(label)
assert len(rejected) == len(mutations)
print('Rejected invalid cases:', ', '.join(rejected))

Rejected invalid cases: missing question, unknown option, bad sum, nonfinite value, incorrect expectation


### 7. Prepare an evaluation request without sending it
The capstone dataset can also become a small real-model evaluation. This cell prepares a request
for one case. Do not accidentally send the answer label or split to the model. A proper live study
would run all selected cases, record costs/errors, and compare against held-out human labels.

In [8]:
from lab_core import load_tickets
evaluation_case = next(r for r in load_tickets() if r['split'] == 'test')
evaluation_request = copy.deepcopy(payload)
evaluation_request['state'] = {'customer_message': evaluation_case['text']}
evaluation_request['questions'] = {'queue': {
    'type': 'choice', 'instructions': 'Select the primary support topic. Broken product behavior takes precedence over the document topic.',
    'criteria': {'billing': 'Charges, invoices, refunds, payment or price questions',
                 'technical': 'Broken application behavior, crashes, failed features',
                 'account': 'Identity, passwords, sign-in, access or email changes'}}}
assert 'label' not in evaluation_request['state']
print('Prepared, NOT SENT:', evaluation_case['id'])
print(json.dumps(evaluation_request, indent=2))

Prepared, NOT SENT: test-037
{
  "model": "jev-1.13.0",
  "state": {
    "customer_message": "The payment receipt lists two charges."
  },
  "questions": {
    "queue": {
      "type": "choice",
      "instructions": "Select the primary support topic. Broken product behavior takes precedence over the document topic.",
      "criteria": {
        "billing": "Charges, invoices, refunds, payment or price questions",
        "technical": "Broken application behavior, crashes, failed features",
        "account": "Identity, passwords, sign-in, access or email changes"
      }
    }
  }
}


**Predict first — exercise 5:** Why is copying the entire dataset row into `state` an evaluation leak?
How would you separate a legitimate policy reference from the hidden answer label?

## Checks
Deliberately corrupt one probability and verify that our integration rejects it.

In [9]:
import copy
broken = copy.deepcopy(fixture)
broken['answers']['refund_requested']['noul'] = 1.4
try:
    validate_response(broken, payload)
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('Invalid probability was accepted')
print('Live call completed:', live_response is not None)

Expected rejection: refund_requested: invalid Noul
Live call completed: False


## Next Steps
Change the fictional message and predict how each answer should move. Then compare the actual response
if you have enabled live access. One successful request validates connectivity, not model quality.

Build a labeled evaluation set before choosing operating thresholds. Preserve the actual model ID,
question definitions, and input version so comparisons are interpretable. Keep exact calculations and
action permissions in code. Return to [README](../README.md) for the full learning path and source list.

## Take three ideas with you

- A request contains the model ID, state, and typed questions.
- Local fixtures, valid responses, and correct judgments are three different things.
- A successful request is the start of evaluation.

**You are ready to move on when:** You can identify exactly what gets sent and tell a handmade fixture from a real model result.

## Check your understanding

The local response validator passes. Did we establish that Jev answered correctly?

Pause and explain your answer before opening the explanation.

<details>
<summary>Show the explanation</summary>
<p>No. In offline mode it checked a handmade fixture. Even a valid live response would establish structure, not semantic correctness.</p>
</details>

For a short next step, try the [practice questions](../docs/PRACTICE.md).
For runnable exercises, use the [worked exercises](08_exercises_and_solutions.ipynb).

If you are unsure where to go next, use [the course outline](../docs/COURSE.md).

[Assignments](../assignments/README.md) · [Quick reference](../docs/QUICK_REFERENCE.md) · [Course outline](../docs/COURSE.md)